# Phase 6: Feature Experiments and Model Optimization

## Objective
Evaluate the impact of feature engineering and optimize
machine learning models using chronological validation.

## Experiments
1. Feature Set Comparison
2. Walk-Forward Cross-Validation
3. Hyperparameter Tuning

In [1]:
# Step 6.1: Feature Set Comparison

from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Locate project
project = Path.cwd()

if not (project / "data_processed").exists():
    project = project.parent

ml_df = pd.read_csv(
    project / "data_processed" / "pjm_rto_ml_features.csv"
)

ml_df["forecast_origin"] = pd.to_datetime(
    ml_df["forecast_origin"], utc=True
)

ml_df["forecast_year"] = (
    ml_df["forecast_origin"]
    .dt.tz_convert("America/New_York")
    .dt.year
)

train = ml_df[ml_df["forecast_year"] <= 2023].copy()
val = ml_df[ml_df["forecast_year"] == 2024].copy()

# Define feature groups
calendar_features = [
    "hour", "day_of_week", "month",
    "is_weekend", "horizon"
]

lag_features = [
    "lag_24", "lag_48", "lag_168"
]

rolling_features = [
    "rolling_mean_24", "rolling_std_24",
    "rolling_mean_168", "rolling_std_168"
]

feature_sets = {
    "A_Calendar": calendar_features,
    "B_Calendar_Lag": calendar_features + lag_features,
    "C_Full": calendar_features + lag_features + rolling_features
}

# Compare feature sets using identical XGBoost parameters
feature_results = []

for name, features in feature_sets.items():

    model = XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train[features], train["mw"])

    predictions = model.predict(val[features])

    rmse = np.sqrt(
        mean_squared_error(val["mw"], predictions)
    )

    mae = mean_absolute_error(val["mw"], predictions)

    feature_results.append({
        "Feature Set": name,
        "Features": len(features),
        "RMSE": rmse,
        "MAE": mae
    })

feature_results = pd.DataFrame(feature_results)

display(feature_results.round(2))

,Feature Set,Features,RMSE,MAE
0,A_Calendar,5,8760.25,6363.17
1,B_Calendar_Lag,8,5469.23,3960.67
2,C_Full,12,5288.81,3824.08


### Feature Set Comparison

Adding historical lag features reduced validation RMSE by 37.6% compared with calendar features alone. Including rolling statistics further reduced RMSE by 3.3%, with the full feature set achieving the lowest RMSE of 5,288.81 MW. These results suggest that historical electricity demand is particularly valuable for day-ahead forecasting.

In [2]:
# Step 6.2: Walk-Forward Cross-Validation

cv_results = []

for val_year in [2022, 2023, 2024]:

    # Training data from earlier years
    train_fold = ml_df[
        ml_df["forecast_year"] < val_year
    ].copy()

    # Validation data from the selected year
    val_fold = ml_df[
        ml_df["forecast_year"] == val_year
    ].copy()

    # Prevent training targets from overlapping validation period
    first_val_origin = val_fold["forecast_origin"].min()

    train_fold = train_fold[
        pd.to_datetime(
            train_fold["datetime_utc"], utc=True
        ) < first_val_origin
    ]

    model = XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fold[feature_sets["C_Full"]],
        train_fold["mw"]
    )

    predictions = model.predict(
        val_fold[feature_sets["C_Full"]]
    )

    rmse = np.sqrt(
        mean_squared_error(val_fold["mw"], predictions)
    )

    mae = mean_absolute_error(
        val_fold["mw"], predictions
    )

    cv_results.append({
        "Validation Year": val_year,
        "Training Rows": len(train_fold),
        "Validation Rows": len(val_fold),
        "RMSE": rmse,
        "MAE": mae
    })

cv_results = pd.DataFrame(cv_results)

display(cv_results.round(2))

print("Mean RMSE:", round(cv_results["RMSE"].mean(), 2))
print("Mean MAE:", round(cv_results["MAE"].mean(), 2))

,Validation Year,Training Rows,Validation Rows,RMSE,MAE
0,2022,8592,8760,5971.89,4182.25
1,2023,17352,8760,4347.74,3199.83
2,2024,26112,8784,5288.81,3824.08


Mean RMSE: 5202.81
Mean MAE: 3735.39


### Walk-Forward Cross-Validation Results

XGBoost achieved a mean RMSE of 5,202.81 MW across three validation years. Performance was strongest in 2023 and weaker in 2022, indicating variation in forecasting accuracy across different periods. Although the training dataset expanded over time, prediction error did not consistently decrease. These results highlight the importance of evaluating machine learning models across multiple chronological validation windows.

In [3]:
# Step 6.3: XGBoost Hyperparameter Tuning

from sklearn.model_selection import ParameterSampler
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

# Candidate hyperparameters
param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5]
}

# Randomly sample 8 combinations
param_list = list(ParameterSampler(
    param_grid,
    n_iter=8,
    random_state=42
))

tuning_results = []

for i, params in enumerate(param_list, start=1):

    fold_rmses = []

    for val_year in [2022, 2023, 2024]:

        train_fold = ml_df[
            ml_df["forecast_year"] < val_year
        ].copy()

        val_fold = ml_df[
            ml_df["forecast_year"] == val_year
        ].copy()

        # Prevent training targets from entering validation period
        first_val_origin = val_fold["forecast_origin"].min()

        target_times = pd.to_datetime(
            train_fold["datetime_utc"], utc=True
        )

        train_fold = train_fold[
            target_times < first_val_origin
        ]

        model = XGBRegressor(
            **params,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train_fold[feature_sets["C_Full"]],
            train_fold["mw"]
        )

        pred = model.predict(
            val_fold[feature_sets["C_Full"]]
        )

        rmse = np.sqrt(
            mean_squared_error(val_fold["mw"], pred)
        )

        fold_rmses.append(rmse)

    tuning_results.append({
        **params,
        "Mean CV RMSE": np.mean(fold_rmses),
        "Std CV RMSE": np.std(fold_rmses)
    })

    print(f"Combination {i}/8 completed")

tuning_results = pd.DataFrame(tuning_results)

tuning_results = tuning_results.sort_values(
    "Mean CV RMSE"
).reset_index(drop=True)

display(tuning_results.round(2))

Combination 1/8 completed
Combination 2/8 completed
Combination 3/8 completed
Combination 4/8 completed
Combination 5/8 completed
Combination 6/8 completed
Combination 7/8 completed
Combination 8/8 completed


,n_estimators,min_child_weight,max_depth,learning_rate,Mean CV RMSE,Std CV RMSE
0,300,1,4,0.05,5099.64,674.58
1,500,3,3,0.03,5110.74,654.39
2,300,3,6,0.03,5114.26,690.95
3,500,1,4,0.03,5130.64,683.23
4,700,5,5,0.05,5238.76,642.20
5,500,3,6,0.05,5255.33,643.06
6,700,3,3,0.10,5326.03,675.94
7,700,1,4,0.10,5364.45,640.93


In [4]:
# Step 6.4: Best XGBoost parameters

best_row = tuning_results.iloc[0]

best_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "min_child_weight": int(best_row["min_child_weight"])
}

print("Best Hyperparameters:")
print(best_params)

print("\nBest Mean CV RMSE:",
      round(best_row["Mean CV RMSE"], 2))

print("Original Mean CV RMSE: 5202.81 MW")

Best Hyperparameters:
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1}

Best Mean CV RMSE: 5099.64
Original Mean CV RMSE: 5202.81 MW


In [5]:
# Step 6.4: Random Forest Hyperparameter Tuning

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error

rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, None],
    "min_samples_leaf": [1, 3, 5],
    "max_features": [0.7, 1.0]
}

rf_param_list = list(ParameterSampler(
    rf_param_grid,
    n_iter=6,
    random_state=42
))

# Include original Random Forest parameters for comparison
original_rf_params = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_leaf": 3,
    "max_features": 1.0
}

rf_param_list.insert(0, original_rf_params)

rf_tuning_results = []

for i, params in enumerate(rf_param_list, start=1):

    fold_rmses = []
    fold_maes = []

    for val_year in [2022, 2023, 2024]:

        train_fold = ml_df[
            ml_df["forecast_year"] < val_year
        ].copy()

        val_fold = ml_df[
            ml_df["forecast_year"] == val_year
        ].copy()

        first_val_origin = val_fold["forecast_origin"].min()

        # Prevent training targets from crossing validation boundary
        target_times = pd.to_datetime(
            train_fold["datetime_utc"], utc=True
        )

        train_fold = train_fold[
            target_times < first_val_origin
        ]

        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train_fold[feature_sets["C_Full"]],
            train_fold["mw"]
        )

        pred = model.predict(
            val_fold[feature_sets["C_Full"]]
        )

        fold_rmses.append(
            np.sqrt(mean_squared_error(val_fold["mw"], pred))
        )

        fold_maes.append(
            mean_absolute_error(val_fold["mw"], pred)
        )

    rf_tuning_results.append({
        **params,
        "Mean CV RMSE": np.mean(fold_rmses),
        "Mean CV MAE": np.mean(fold_maes),
        "Std CV RMSE": np.std(fold_rmses)
    })

    print(f"Random Forest combination {i}/{len(rf_param_list)} completed")

rf_tuning_results = pd.DataFrame(rf_tuning_results)

rf_tuning_results = rf_tuning_results.sort_values(
    "Mean CV RMSE"
).reset_index(drop=True)

display(rf_tuning_results.round(2))

Random Forest combination 1/7 completed
Random Forest combination 2/7 completed
Random Forest combination 3/7 completed
Random Forest combination 4/7 completed
Random Forest combination 5/7 completed
Random Forest combination 6/7 completed
Random Forest combination 7/7 completed


,n_estimators,max_depth,min_samples_leaf,max_features,Mean CV RMSE,Mean CV MAE,Std CV RMSE
0,300,NaN,5,0.7,5239.23,3774.18,641.90
1,100,15.0,1,0.7,5270.02,3797.87,620.27
2,200,10.0,3,0.7,5306.21,3838.13,618.63
3,100,10.0,1,0.7,5315.22,3836.20,626.92
4,300,20.0,3,1.0,5334.97,3845.49,625.71
5,200,15.0,3,1.0,5336.24,3848.03,620.47
6,200,15.0,1,1.0,5357.22,3862.09,618.67


In [6]:
# Step 6.5: Best Random Forest Configuration

rf_best_row = rf_tuning_results.iloc[0]

rf_best_params = {
    "n_estimators": int(rf_best_row["n_estimators"]),
    "max_depth": (
        None if pd.isna(rf_best_row["max_depth"])
        else int(rf_best_row["max_depth"])
    ),
    "min_samples_leaf": int(rf_best_row["min_samples_leaf"]),
    "max_features": float(rf_best_row["max_features"])
}

print("Best Random Forest Parameters:")
print(rf_best_params)

print("\nBest RF Mean CV RMSE:",
      round(rf_best_row["Mean CV RMSE"], 2))

print("Best XGBoost Mean CV RMSE: 5099.64 MW")

Best Random Forest Parameters:
{'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5, 'max_features': 0.7}

Best RF Mean CV RMSE: 5239.23
Best XGBoost Mean CV RMSE: 5099.64 MW


In [7]:
# Step 6.6: Optimized Model Comparison

optimized_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Mean CV RMSE": rf_best_row["Mean CV RMSE"],
        "Std CV RMSE": rf_best_row["Std CV RMSE"]
    },
    {
        "Model": "XGBoost",
        "Mean CV RMSE": best_row["Mean CV RMSE"],
        "Std CV RMSE": best_row["Std CV RMSE"]
    }
])

optimized_results = optimized_results.sort_values(
    "Mean CV RMSE"
).reset_index(drop=True)

display(optimized_results.round(2))

selected_model = optimized_results.iloc[0]["Model"]

print("\nSelected Model:", selected_model)
print("Best XGBoost Parameters:", best_params)
print("Best Random Forest Parameters:", rf_best_params)

,Model,Mean CV RMSE,Std CV RMSE
0,XGBoost,5099.64,674.58
1,Random Forest,5239.23,641.90



Selected Model: XGBoost
Best XGBoost Parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1}
Best Random Forest Parameters: {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5, 'max_features': 0.7}


In [8]:
# Step 9.2.4: Export ML experiment results

from pathlib import Path

project = Path.cwd()

if not (project / "results").exists():
    project = project.parent

tables_dir = project / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

experiment_tables = {
    "feature_set_comparison.csv": feature_results,
    "walk_forward_cv.csv": cv_results,
    "xgboost_tuning.csv": tuning_results,
    "random_forest_tuning.csv": rf_tuning_results,
    "optimized_model_comparison.csv": optimized_results
}

for filename, table in experiment_tables.items():
    table.to_csv(tables_dir / filename, index=False)
    print(f"Saved: {filename}")

print("\nAll ML experiment results exported successfully!")

Saved: feature_set_comparison.csv
Saved: walk_forward_cv.csv
Saved: xgboost_tuning.csv
Saved: random_forest_tuning.csv
Saved: optimized_model_comparison.csv

All ML experiment results exported successfully!
